# Stage 6 · Multi-Turn, Tools & Environment RL — EXERCISES
### Topics: Trajectory MDPs · Tool Actions · Budgets · POMDP-style Observations · Rollouts · Credit Assignment

> Fill every `# TODO`. Run `# ASSERT` cells to verify.


In [ ]:
import json
import re
from dataclasses import dataclass, field
from typing import Any, Callable, Dict, List, Optional, Tuple

import numpy as np


---
## 1 · Episodes, transitions & Monte Carlo returns

Given per-step rewards $(r_0,\dots,r_{T-1})$, compute backward **Monte Carlo returns** $G_t = r_t + \gamma G_{t+1}$.

`total_return_from_rewards` should return $G_0$ (scalar).


In [ ]:
@dataclass
class Transition:
    observation: str
    action:      str
    reward:      float
    done:        bool
    info:        Dict[str, Any] = field(default_factory=dict)


def compute_mc_returns(rewards: List[float], gamma: float) -> List[float]:
    """Backward pass: G_t = r_t + γ G_{t+1}. Length equals len(rewards)."""
    # TODO
    raise NotImplementedError


def total_return_from_rewards(rewards: List[float], gamma: float) -> float:
    """Return G_0; 0.0 if rewards is empty."""
    # TODO
    raise NotImplementedError


# ── ASSERT ────────────────────────────────────────────────────────────────
assert compute_mc_returns([1.0, 2.0, 3.0], 0.0) == [1.0, 2.0, 3.0]
assert abs(compute_mc_returns([1.0, 2.0, 3.0], 1.0)[0] - 6.0) < 1e-6
g = compute_mc_returns([0.0, 0.0, 1.0], 1.0)
assert abs(g[0] - 1.0) < 1e-6 and abs(g[-1] - 1.0) < 1e-6
assert total_return_from_rewards([], 0.9) == 0.0
print("compute_mc_returns ✓  total_return_from_rewards ✓")


---
## 2 · Parsing tool calls and final answers

- `parse_tool_json`: if stripped action starts with `TOOL` (case-insensitive), skip the first 4 characters of the **original** stripped string for the JSON payload (`json.loads` on `action.strip()[4:].strip()` after verifying the prefix). On failure return `None`. If nothing after `TOOL`, return `None`.
- `parse_answer_float`: find `ANSWER` (case-insensitive) then capture a float (regex similar to solution).


In [ ]:
def parse_tool_json(action: str) -> Optional[Dict[str, Any]]:
    # TODO
    raise NotImplementedError


def parse_answer_float(action: str) -> Optional[float]:
    # TODO
    raise NotImplementedError


# ── ASSERT ────────────────────────────────────────────────────────────────
assert parse_tool_json('TOOL {"name":"lookup","key":"banana"}') == {"name": "lookup", "key": "banana"}
assert parse_tool_json("TOOL") is None
assert parse_tool_json("say TOOL {}") is None
assert abs(parse_answer_float("  ANSWER   0.5  ") - 0.5) < 1e-9
print("parse_tool_json ✓  parse_answer_float ✓")


---
## 3 · `ToyPricingEnv`

Implement the environment described in the solution notebook overview:
- Hidden prices: `apple=1.2`, `banana=0.5`, `cherry=2.0`; target key **`banana`**.
- `reset(question=None)`: default question `"What is the price of banana (USD)?"`. Include lines telling the agent how to use `TOOL {"name":"lookup","key":...}` and `ANSWER <number>`.
- If `pomdp` is **False**, append a line `ORACLE: valid keys are ...` with sorted keys.
- `step(action)`:
  - Increment an internal turn counter at the **start** of each step. If turns exceed `max_turns`, return truncated done with observation explaining truncation.
  - **TOOL** JSON: `name` must be `"lookup"` and include `"key"`. Successful lookup of a known key yields observation `Tool result: <key> = <value>` and reward `tool_reward` (default **0.1**). Unknown key → reward `0.0`.
  - Enforce `max_tool_calls`: if a TOOL arrives when budget already used, return done with negative reward `-0.05` and `info` containing `truncated=True`, `reason="tool_budget"`.
  - **ANSWER**: if within `1e-5` of true price → reward `1.0`, `done=True`, `info["terminated"]=True`, `info["correct"]=True`.
  - Wrong answer → reward `-0.05`, not done.
  - Malformed / unrecognised → reward `-0.01`, not done.

Use `parse_tool_json` / `parse_answer_float` from §2.


In [ ]:
class ToyPricingEnv:
    def __init__(
        self,
        pomdp:          bool = True,
        max_turns:      int  = 8,
        max_tool_calls: int  = 3,
        tool_reward:    float = 0.1,
    ):
        self.pomdp = pomdp
        self.max_turns = max_turns
        self.max_tool_calls = max_tool_calls
        self.tool_reward = tool_reward
        self._prices = {"apple": 1.2, "banana": 0.5, "cherry": 2.0}
        self._target_key = "banana"
        self._answer = self._prices[self._target_key]
        self._turn = 0
        self._tool_calls = 0
        self._last_obs = ""

    def reset(self, question: Optional[str] = None) -> str:
        # TODO
        raise NotImplementedError

    def step(self, action: str) -> Tuple[str, float, bool, Dict[str, Any]]:
        # TODO
        raise NotImplementedError


# ── ASSERT ────────────────────────────────────────────────────────────────
env = ToyPricingEnv(pomdp=True)
obs0 = env.reset()
assert "banana" in obs0.lower() and "ORACLE" not in obs0
obs1, r1, d1, i1 = env.step('TOOL {"name":"lookup","key":"banana"}')
assert not d1 and abs(r1 - 0.1) < 1e-9 and "0.5" in obs1
obs2, r2, d2, i2 = env.step("ANSWER 0.5")
assert d2 and abs(r2 - 1.0) < 1e-9 and i2.get("terminated") and i2.get("correct")

env_o = ToyPricingEnv(pomdp=False)
assert "ORACLE" in env_o.reset() and "apple" in env_o.reset()

env_b = ToyPricingEnv(pomdp=True, max_tool_calls=1)
env_b.reset()
env_b.step('TOOL {"name":"lookup","key":"banana"}')
obs_b, r_b, d_b, i_b = env_b.step('TOOL {"name":"lookup","key":"apple"}')
assert d_b and i_b.get("truncated") and i_b.get("reason") == "tool_budget"

print("ToyPricingEnv ✓")


---
## 4 · Rollouts

`rollout_episode(env, policy, max_steps=16)`:
- Call `env.reset()`; maintain `history` list starting with that observation.
- Loop: `action = policy(history)`; `env.step(action)`; record a `Transition` with the **pre-step** observation stored in the transition (same pattern as solution).
- Append new observations to `history` when not done.

`scripted_policy(actions)` returns a `PolicyFn` that consumes the next string from `actions` each call.


In [ ]:
PolicyFn = Callable[[List[str]], str]


def rollout_episode(
    env:       ToyPricingEnv,
    policy:    PolicyFn,
    max_steps: int = 16,
) -> List[Transition]:
    # TODO
    raise NotImplementedError


def scripted_policy(actions: List[str]) -> PolicyFn:
    # TODO
    raise NotImplementedError


# ── ASSERT ────────────────────────────────────────────────────────────────
tr = rollout_episode(
    ToyPricingEnv(pomdp=True),
    scripted_policy(['TOOL {"name":"lookup","key":"banana"}', "ANSWER 0.5"]),
)
assert len(tr) == 2 and tr[-1].done and abs(tr[-1].reward - 1.0) < 1e-9
print("rollout_episode ✓  scripted_policy ✓")


---
## 5 · Centered Monte Carlo advantages

Given per-step rewards, compute MC returns $G_t$, then **advantages** $A_t = G_t - \text{mean}(G)$.

`all_close_zero(xs, eps=1e-9)` returns whether every $|x|<\varepsilon$.

**Check:** for sparse terminal reward `[0,0,1]` with $\gamma=1$, all $G_t$ are `1` → centered advantages are all zero.


In [ ]:
def centered_mc_advantages(rewards: List[float], gamma: float) -> List[float]:
    # TODO: reuse compute_mc_returns
    raise NotImplementedError


def all_close_zero(xs: List[float], eps: float = 1e-9) -> bool:
    # TODO
    raise NotImplementedError


# ── ASSERT ────────────────────────────────────────────────────────────────
adv_sparse = centered_mc_advantages([0.0, 0.0, 1.0], gamma=1.0)
assert all_close_zero(adv_sparse), "Terminal-only + γ=1 → zero centered advantages"

adv_shaped = centered_mc_advantages([0.1, 0.1, 1.0], gamma=1.0)
assert not all_close_zero(adv_shaped)
assert abs(np.mean(adv_shaped)) < 1e-6
print("centered_mc_advantages ✓  all_close_zero ✓")


---
## 6 · One-step REINFORCE on a softmax macro-policy (NumPy)

Policy $\pi=\mathrm{softmax}(\theta)$. For sampled action $a$ and scalar advantage $A$:
$$\nabla_\theta \log \pi(a) = e_a - \pi$$
Take one gradient-descent step on $L=-A\log\pi(a)$:
$$\theta \leftarrow \theta + \eta \, A \, (e_a - \pi)$$

Implement `macro_bandit_reinforce_step(logits, action_idx, advantage, lr)` updating `logits` **in place** and returning the scalar loss **before** the update: $-A\log\pi(a)$.


In [ ]:
def macro_bandit_reinforce_step(
    logits:     np.ndarray,
    action_idx: int,
    advantage:  float,
    lr:         float = 0.1,
) -> float:
    # TODO
    raise NotImplementedError


# ── ASSERT ────────────────────────────────────────────────────────────────
np.random.seed(0)
lg = np.zeros(3, dtype=np.float64)
loss0 = macro_bandit_reinforce_step(lg, action_idx=1, advantage=1.0, lr=0.5)
assert isinstance(loss0, float)
ex = np.exp(lg - lg.max())
p = ex / ex.sum()
assert p[1] > 1.0 / 3.0
print("macro_bandit_reinforce_step ✓")
